In [366]:
# import
import torch
import random

import warnings
warnings.filterwarnings("ignore")

seed = 333

# Data

## Import Data

In [367]:
# import data
with open("../data/input.txt","r",newline="\n") as file :
    data_file = file.read()

data_file[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

## Data Class

In [370]:
class Data:
    def __init__(
        self,
        _data: str,
        seed: int = seed,
        train_batch_size: int = None,
        train_block_size: int = None,
        test_batch_size: int = None,
        test_block_size: int = None,
    ) -> None:
        # initialisation
        self.seed = seed
        self.data = _data  # raw data without process
        self.len_data = len(_data)  # how many character in raw data
        self.char_list = sorted(list(set(_data)))  # list of possible character
        self.len_char = len(self.char_list)  # len of possible character

        # encoder and decoder list
        self.enc_dict = {key: value for value, key in enumerate(self.char_list)}  # convert to number
        self.dec_dict = {key: value for key, value in enumerate(self.char_list)}  # convert to string

        # encoded data
        self.data_enc = self.encoder(self.data)  # raw data without process

        # split to train and test set with ration 9:1
        split_at_index = self.len_data // 9
        self.train = self.data_enc[:split_at_index]  # full train set
        self.test = self.data_enc[split_at_index:]  # full test set

        # create x,y for train and test set
        if not None in [train_batch_size, test_batch_size, train_block_size, test_block_size]:
            self.x_train, self.y_train = self.sample_batch(
                batch_size=train_batch_size, block_size=train_block_size, input=self.train
            )
            self.x_test, self.y_test = self.sample_batch(
                batch_size=test_batch_size, block_size=test_block_size, input=self.test
            )

    def encoder(self, input: str) -> torch.Tensor:
        """convert string to encoded index

        Args:
            input (str): string to be encoded

        Returns:
            torch.Tensor: (copy) tensor of encoded index
        """
        return torch.tensor([self.enc_dict[char] for char in input])

    def decoder(self, input: torch.Tensor) -> str:
        """covnert encoded index in torch.tensor into

        Args:
            input (torch.Tensor): tensor of encoded index

        Returns:
            str: (copy) string from encoded index in tensor(input)
        """
        return "".join([self.dec_dict[i.item()] for i in input])

    def sample_batch(self, batch_size: int, block_size: int, input: torch.Tensor):
        """generate sample batch

        Args:
            batch_size (int): how many batch generated
            block_size (int): maximum length per batch
            input (torch.Tensor): tensor of encoded data

        Returns:
            x,y: data_x,data_y
        """
        random.seed(self.seed)
        output_x = torch.empty((batch_size, block_size), dtype=torch.long)
        output_y = torch.empty((batch_size, block_size), dtype=torch.long)
        len_input = len(input)

        for batch_index in range(batch_size):
            offset = random.randint(0, len_input - block_size - 1)
            output_x[batch_index] = input[offset : offset + block_size]
            output_y[batch_index] = input[offset + 1 : offset + block_size + 1]
        return output_x, output_y

In [371]:
d = Data(
    _data=data_file,
    seed=seed,
    train_batch_size=None,
    train_block_size=None,
    test_batch_size=None,
    test_block_size=None,
)